# Download Common Voice 26.0 (Swahili) from Mozilla Data Collective

Uses the official `datacollective` Python library to authenticate with and
download dataset `cmqim4c1000tmnr07zq3vwhor`
(slug `common-voice-scripted-speech-26-0-swahil-0228b2f6`) from the Mozilla
Data Collective (MDC) API, then selects and extracts only the audio
actually needed rather than the full ~20GB+ `clips/` directory.

**Flow:** install -> set API key -> download archive -> extract just the
transcript metadata -> select a phoneme-balanced subset -> extract only
the selected clips. Full corpus audio is never extracted in bulk, since
that alone does not fit most Kaggle sessions' disk.

The Dataset ID and Dataset Slug are interchangeable everywhere below -- use
whichever you find easier to keep track of.

**Before running, in this notebook's settings:** Settings -> Internet -> On
(required; MDC and GitHub are external).

## 1. Install the `datacollective` package

In [ ]:
!pip install -q datacollective

## 2. Set your API key as an environment variable

Generate a key at https://mozilladatacollective.com after creating an account
and agreeing to this dataset's Terms & Conditions on its MDC page.

**Recommended on Kaggle:** store it as a Kaggle Secret (Add-ons -> Secrets)
named `MDC_API_KEY` and load it at runtime as below, rather than typing the
raw key into a cell -- notebook cells and their outputs can end up saved,
shared, or version-controlled.
*If this key was ever pasted anywhere outside a secrets vault (chat, a
script, a committed file), rotate/revoke it on the MDC platform first.*

Outside Kaggle, the library also reads `MDC_API_KEY` from a `.env` file in
the working directory (via `python-dotenv`) or from the shell environment --
either works with the exact same `download_dataset`/`load_dataset` calls
below, no code changes needed.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["MDC_API_KEY"] = user_secrets.get_secret("MDC_API_KEY")
print("MDC_API_KEY loaded from Kaggle Secrets.")

## 3. Download the archive

`download_dataset` fetches the raw archive to local disk and returns its path
(resumes automatically if re-run after an interruption).

In [ ]:
from datacollective import download_dataset

DATASET_ID = "cmqim4c1000tmnr07zq3vwhor"  # or the slug: "common-voice-scripted-speech-26-0-swahil-0228b2f6"
# Explicit directory: the SDK's own default (~/.mozdata/datasets, i.e.
# /root/.mozdata/datasets on Kaggle) lives outside every Kaggle-persisted
# area and is wiped when the session ends.
DOWNLOAD_DIR = "/kaggle/working/mdc_common_voice_sw"

dataset_path = download_dataset(DATASET_ID, download_directory=DOWNLOAD_DIR)
print("Downloaded archive to:", dataset_path)

Optional: `load_dataset` downloads (if needed), extracts, and parses the
dataset via MDC's schema registry, returning a pandas DataFrame directly --
if this dataset has a registered schema. If not, it raises `RuntimeError`;
either way, the metadata-only extraction below is the path the rest of this
notebook relies on.

In [ ]:
from datacollective import load_dataset

try:
    dataset = load_dataset(DATASET_ID, download_directory=DOWNLOAD_DIR)
    print(dataset.shape)
    display(dataset.head())
except RuntimeError as e:
    print("No registered schema for this dataset yet -- continuing with the metadata extraction below instead:")
    print(e)

## 4. Extract just the transcript metadata

Full Common Voice audio (`clips/`) is far larger than most Kaggle sessions
can hold alongside the compressed archive -- extracting everything up front
is what causes `OSError: No space left on device`. `select_subset.py` only
needs the transcript TSV (a few MB), so extract just that for now; audio for
only the *selected* utterances is extracted later in step 6, after selection
narrows ~700k+ clips down to a target subset (e.g. 10,000).

In [ ]:
import tarfile
from pathlib import Path

metadata_dir = Path("/kaggle/working/cv26_sw_metadata")
metadata_dir.mkdir(parents=True, exist_ok=True)

extracted = []
with tarfile.open(dataset_path) as tar:
    for member in tar:
        if member.isfile() and member.name.endswith(".tsv"):
            member.name = Path(member.name).name  # flatten any archive subdirectory structure
            tar.extract(member, path=metadata_dir)
            extracted.append(member.name)

print(f"Extracted {len(extracted)} TSV file(s) to {metadata_dir}")
for name in sorted(extracted):
    print(" -", name)

This reads through the whole compressed archive once (unavoidable for a
single tar.gz), but only ever writes the small TSVs to disk -- it will not
hit the earlier disk error.

## 5. Select a phoneme-balanced, speaker-diverse subset

Clones the Swahili-Deepfake-dataset pipeline repo and runs
`scripts/select_subset.py` (see `docs/METHODOLOGY.md` there) to reduce
~700k+ Common Voice utterances down to a phoneme-balanced target subset --
e.g. 10,000 utterances -- before any audio is extracted.

If this repo is private for you, cloning will fail; in that case copy
`scripts/select_subset.py`, `scripts/extract_selected_clips.py`, and the
`swahili_deepfake_dataset/` package into this session another way (e.g. a
Kaggle Dataset containing the repo) instead of git-cloning it.

In [ ]:
!git clone --depth 1 https://github.com/regak/Swahili-Deepfake-dataset.git /kaggle/working/Swahili-Deepfake-dataset

In [ ]:
validated_tsv = metadata_dir / "validated.tsv"
if not validated_tsv.exists():
    candidates = sorted(metadata_dir.glob("*.tsv"))
    print("validated.tsv not found; available TSVs:", [p.name for p in candidates])
    if candidates:
        validated_tsv = candidates[0]
print("Using:", validated_tsv)

In [ ]:
SELECTED_SUBSET_TSV = "/kaggle/working/selected_subset.tsv"

!python /kaggle/working/Swahili-Deepfake-dataset/scripts/select_subset.py \
    "{validated_tsv}" \
    --target-size 10000 --max-per-speaker 100 \
    --output "{SELECTED_SUBSET_TSV}" \
    --report /kaggle/working/phoneme_coverage_report.json

## 6. Extract only the selected clips

Extracts just the few thousand selected utterances' audio directly from the
original archive -- not the full corpus -- using
`scripts/extract_selected_clips.py`.

In [ ]:
CLIPS_DIR = "/kaggle/working/clips"

!python /kaggle/working/Swahili-Deepfake-dataset/scripts/extract_selected_clips.py \
    "{dataset_path}" \
    "{SELECTED_SUBSET_TSV}" \
    "{CLIPS_DIR}"

## Disk space and persisting across sessions

By selecting a subset before extracting audio, total disk usage stays close
to `--target-size` clips (a few GB) plus the ~20-22 GB compressed archive --
much less than fully extracting Common Voice's ~700k+ clips would need.

Once step 6 has finished, the archive itself is no longer needed and can be
removed to reclaim that ~20-22 GB:
```python
# import os
# os.remove(dataset_path)
```

**Nothing outside `/kaggle/working` (via Save Version) or an attached
`/kaggle/input` Dataset survives when a session ends** -- including the
SDK's own default cache directory (`~/.mozdata/datasets`) and `/kaggle/temp`.
To avoid re-running this whole notebook in every future session:

1. Let this notebook finish (metadata extraction, selection, selected-clip extraction).
2. Delete `dataset_path` (see above) to free the ~20-22 GB archive.
3. Click **Save Version** to commit the notebook and its `/kaggle/working`
   output -- now small enough (selected clips + TSVs) to comfortably fit
   the output quota.
4. From the notebook's output/Data pane, use **"New Dataset"** to publish
   that output as a private Kaggle Dataset.
5. In any future notebook, **Add Input -> your new dataset** -- it mounts
   read-only at `/kaggle/input/<dataset-slug>/` instantly, no MDC download
   or re-extraction needed.